# 05 - Publication Tables & Figures

Formats Notebook 04's outputs into journal-ready tables (LaTeX + Markdown)
and figures (matplotlib, vector-friendly, consistent styling), following the
table/figure conventions used in Guo et al. (2025) Neural Computing and
Applications (error tables with best-result highlighting; horizon-vs-error
line plots; model-size/complexity plots).

## Colab setup
Run this once per session. It mounts Google Drive (where the whole `potato_pipeline/` project folder should be uploaded), `cd`s into `notebooks/` so every relative path (`../src`, `../data`, `../results`) keeps working unchanged, and installs the extra packages this notebook needs (Colab's base image doesn't include them).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust this if you uploaded the project folder somewhere else in Drive
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
import sys, os, json
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = "../results"
FIG_DIR = "../results/figures"
TAB_DIR = "../results/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

HORIZONS = [1, 7, 14, 30]

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

metrics_table = pd.read_csv(f"{RESULTS_DIR}/test_metrics_full.csv")
dm_table = pd.read_csv(f"{RESULTS_DIR}/dm_significance_tests.csv")
metrics_table.head()

## 1. Main results table (MSE, MAPE by model x horizon)

Formatted as mean (std) following common practice for multi-seed DL
benchmark reporting; best result per horizon is bolded.

In [ ]:
def build_main_table(metrics_table, metric="MSE"):
    pivot_mean = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_mean")
    pivot_std = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_std")

    formatted = pivot_mean.copy().astype(object)
    for col in pivot_mean.columns:
        best_idx = pivot_mean[col].idxmin()
        for idx in pivot_mean.index:
            mean_v, std_v = pivot_mean.loc[idx, col], pivot_std.loc[idx, col]
            cell = f"{mean_v:.4f} ({std_v:.4f})"
            if idx == best_idx:
                cell = f"**{cell}**"
            formatted.loc[idx, col] = cell
    formatted.columns = [f"h={c}" for c in formatted.columns]
    return formatted

mse_table = build_main_table(metrics_table, "MSE")
mape_table = build_main_table(metrics_table, "MAPE")

mse_table.to_csv(f"{TAB_DIR}/main_results_mse.csv")
mape_table.to_csv(f"{TAB_DIR}/main_results_mape.csv")

print("MSE (mean (std)), best per horizon in bold (markdown-style **):")
mse_table

In [ ]:
def to_latex_booktabs(df, caption, label):
    latex = df.to_latex(escape=False, column_format="l" + "c" * df.shape[1])
    latex = latex.replace("\\toprule", "\\toprule").replace("**", "\\textbf{").replace("\\textbf{", "\\textbf{", 1)
    # simple ** -> \textbf{...} conversion
    import re
    def bold_repl(m):
        return f"\\textbf{{{m.group(1)}}}"
    latex = re.sub(r"\*\*(.*?)\*\*", bold_repl, latex)
    wrapped = (
        "\\begin{table}[htbp]\n\\centering\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        + latex +
        "\\end{table}"
    )
    return wrapped

with open(f"{TAB_DIR}/main_results_mse.tex", "w") as f:
    f.write(to_latex_booktabs(mse_table, "Test-set MSE (mean (std) across 5 seeds) by model and forecast horizon. Best result per horizon in bold.", "tab:mse_results"))
with open(f"{TAB_DIR}/main_results_mape.tex", "w") as f:
    f.write(to_latex_booktabs(mape_table, "Test-set MAPE (\\%%) (mean (std) across 5 seeds) by model and forecast horizon. Best result per horizon in bold.", "tab:mape_results"))
print("Saved LaTeX tables to", TAB_DIR)

## 2. Diebold-Mariano significance -- full pairwise heatmaps

One heatmap per horizon: cell (row, column) is shaded if the row model's squared error is significantly lower than the column model's, after Holm-Bonferroni correction. This replaces an ELinear-only summary, since Table 3 showed no model dominates every horizon -- the full picture matters more here than a single model's win/loss record. The older ELinear-vs-rest table (`dm_significance_tests.csv`) is still written by Notebook 04 for backward compatibility, in case any manuscript text already references it directly.

In [ ]:
dm_pairwise = pd.read_csv(f"{RESULTS_DIR}/dm_pairwise_full.csv")
all_models_order = ["Naive", "SeasonalNaive", "ARIMA", "LSTM", "TCN", "NBEATS",
                     "DLinear", "NLinear", "PatchTST", "iTransformer", "ELinear"]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, h in zip(axes.flat, HORIZONS):
    sub = dm_pairwise[dm_pairwise.horizon == h]
    models_h = [m for m in all_models_order if m in set(sub.model_a) | set(sub.model_b)]
    n = len(models_h)
    # matrix[i,j] = +1 if row model significantly better than column model,
    #              -1 if column model significantly better than row model,
    #               0 if not significant
    matrix = np.zeros((n, n))
    for _, row in sub.iterrows():
        if not row.significant:
            continue
        i, j = models_h.index(row.model_a), models_h.index(row.model_b)
        val = 1 if row.better_model == row.model_a else -1
        matrix[i, j] = val
        matrix[j, i] = -val

    im = ax.imshow(matrix, cmap="RdBu", vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(models_h, rotation=90, fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(models_h, fontsize=7)
    ax.set_title(f"h = {h} d")

fig.suptitle("Pairwise DM significance (blue = row model significantly better, red = column model significantly better)", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/dm_pairwise_heatmap.pdf")
plt.savefig(f"{FIG_DIR}/dm_pairwise_heatmap.png")
plt.show()

# Also keep the older ELinear-focused table for anyone referencing it directly
dm_table = pd.read_csv(f"{RESULTS_DIR}/dm_significance_tests.csv")
dm_summary = dm_table.pivot(index="benchmark", columns="horizon", values="elinear_significantly_better")
dm_pvals = dm_table.pivot(index="benchmark", columns="horizon", values="p_value_holm")
dm_display = dm_summary.copy().astype(object)
for col in dm_summary.columns:
    for idx in dm_summary.index:
        sig = dm_summary.loc[idx, col]
        p = dm_pvals.loc[idx, col]
        marker = "check (p={:.3f})".format(p) if sig else "n.s. (p={:.3f})".format(p)
        dm_display.loc[idx, col] = marker
dm_display.columns = [f"h={c}" for c in dm_display.columns]
dm_display.to_csv(f"{TAB_DIR}/dm_significance_summary.csv")
dm_display

## 3. Figure: Error vs. forecast horizon, per model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for model_name, grp in metrics_table.groupby("model"):
    grp = grp.sort_values("horizon")
    ax.plot(grp["horizon"], grp["MSE_mean"], marker="o", label=model_name, lw=1.8 if model_name == "ELinear" else 1.2,
            zorder=3 if model_name == "ELinear" else 2)
    ax.fill_between(grp["horizon"], grp["MSE_mean"] - grp["MSE_std"], grp["MSE_mean"] + grp["MSE_std"], alpha=0.12)

ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Test MSE")
ax.set_title("Agra Potato Price: Forecast Error vs. Horizon")
ax.set_xticks([1, 7, 14, 30])
ax.legend(loc="upper left", fontsize=8, ncol=2, frameon=False)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/error_vs_horizon.pdf")
plt.savefig(f"{FIG_DIR}/error_vs_horizon.png")
plt.show()

## 4. Figure: Prediction vs. actual, all horizons and all comparable-scale models

Combined into a single 2x2 figure (one panel per horizon), matching the
layout convention used for Figure 3 below. ARIMA and Seasonal-Naive are
excluded from this overlay for visual scale (their error is roughly 200x
larger than every other model at every horizon; see Section 3.3's caveat) --
including them would flatten every other model's line to the baseline.
ELinear is rendered with a heavier line weight, consistent with it being the
model proposed in this study; this is a rendering choice only and does not
alter any underlying value. Colors are drawn from a colorblind-safe
qualitative palette (Wong 2011, Nature Methods) rather than matplotlib's
default `tab10`, since 9 overlapping series is enough categories that
default palettes commonly used for far fewer lines cannot be assumed to
remain distinguishable under color-vision deficiency.

In [ ]:
PLOT_MODELS = ["Naive", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer", "ELinear"]

# Wong (2011) colorblind-safe qualitative palette (Nature Methods), 8 colors
# + one extra (dark red) to cover the 9th model, ELinear given its own
# strong black-adjacent treatment below regardless of this palette slot.
WONG_PALETTE = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442",
    "#0072B2", "#D55E00", "#CC79A7", "#999999", "#8B0000",
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False)
legend_handles, legend_labels = None, None

for ax, h in zip(axes.flat, HORIZONS):
    path = f"{RESULTS_DIR}/pred_vs_actual_h{h}.csv"
    if not os.path.exists(path):
        ax.set_title(f"h = {h} d (no data -- run Notebook 04's export step first)")
        continue

    df_h = pd.read_csv(path, parse_dates=["date"])
    actual_line, = ax.plot(df_h["date"], df_h["actual"], label="Actual",
                            color="black", lw=2.0, zorder=100)

    handles = [actual_line]
    for model_name, color in zip(PLOT_MODELS, WONG_PALETTE):
        if model_name not in df_h.columns:
            continue
        is_elinear = (model_name == "ELinear")
        line, = ax.plot(
            df_h["date"], df_h[model_name], label=model_name,
            color=("black" if is_elinear else color),
            lw=(2.2 if is_elinear else 1.1),
            linestyle=("--" if is_elinear else "-"),
            alpha=(0.95 if is_elinear else 0.8),
            zorder=(90 if is_elinear else 10),
        )
        handles.append(line)

    ax.set_xlabel("Date")
    ax.set_ylabel("Rs/quintal")
    ax.set_title(f"h = {h} d")
    ax.tick_params(axis="x", rotation=30)

    if legend_handles is None:
        legend_handles = handles
        legend_labels = [h_.get_label() for h_ in handles]

fig.suptitle("Agra Potato Price: Forecast vs. Actual, by Horizon", y=1.02)
fig.legend(legend_handles, legend_labels, loc="lower center",
           ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/pred_vs_actual_all_horizons.pdf", bbox_inches="tight")
plt.savefig(f"{FIG_DIR}/pred_vs_actual_all_horizons.png", bbox_inches="tight")
plt.show()

## 5. Figure: Model complexity vs. accuracy (parameter count vs. MSE), all horizons

Parameter count is **not constant across horizons** for most architectures here -- the output layer scales with `pred_len` (verified in Notebook 04: ELinear alone scales from 87 to 2,610 parameters between h = 1 d and h = 30 d). A single-horizon complexity plot would therefore be misleading about model size at other horizons; one panel per horizon is shown instead. Mirrors Fig. 4 in Guo et al. (2025) -- checks whether the 'scale effect' (bigger = better) reported in NLP/CV holds for this task.

In [ ]:
# Parameter counts are read PER HORIZON from Notebook 04's output
# (param_counts_by_horizon.json) -- no manual entry, avoids transcription
# errors, and correctly reflects that model size changes with pred_len.
with open(f"{RESULTS_DIR}/param_counts_by_horizon.json") as f:
    param_counts_by_horizon = json.load(f)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, h in zip(axes.flat, HORIZONS):
    subset = metrics_table[metrics_table["horizon"] == h].copy()
    param_map = {m: counts.get(str(h)) for m, counts in param_counts_by_horizon.items()}
    subset["params"] = subset["model"].map(param_map)
    subset = subset.dropna(subset=["params"])

    if len(subset) > 0:
        ax.scatter(subset["params"], subset["MSE_mean"])
        for _, row in subset.iterrows():
            ax.annotate(row["model"], (row["params"], row["MSE_mean"]), fontsize=7,
                         xytext=(4, 4), textcoords="offset points")
        ax.set_xscale("log")
        ax.set_xlabel("Parameter count (log scale)")
        ax.set_ylabel("Test MSE")
        ax.set_title(f"h = {h} d")
    else:
        ax.set_title(f"h = {h} d (no data)")

fig.suptitle("Model Complexity vs. Accuracy, by Horizon")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/complexity_vs_accuracy_all_horizons.pdf")
plt.savefig(f"{FIG_DIR}/complexity_vs_accuracy_all_horizons.png")
plt.show()

## 6. Figure: Directional accuracy / practical trading relevance (1-day horizon)

In [ ]:
dir_acc = metrics_table[metrics_table["horizon"] == 1][["model", "DirAcc_mean", "DirAcc_std"]].sort_values("DirAcc_mean", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dir_acc["model"], dir_acc["DirAcc_mean"], yerr=dir_acc["DirAcc_std"], capsize=3)
ax.axhline(50, color="grey", ls="--", lw=1, label="Chance level (50%)")
ax.set_ylabel("Directional accuracy (%)")
ax.set_title("1-day-ahead Directional Accuracy by Model")
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/directional_accuracy_h1.pdf")
plt.show()

## 7. Export a consolidated results package for the manuscript

Bundles the CSV/LaTeX tables and PDF/PNG figures into one folder for easy
upload to a supplementary-materials repository (e.g., alongside a Figshare
DOI, matching the data-availability convention used in Guo et al. 2025).

In [ ]:
import shutil
package_dir = f"{RESULTS_DIR}/manuscript_package"
os.makedirs(package_dir, exist_ok=True)
for src_dir in [TAB_DIR, FIG_DIR]:
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(package_dir, fname))
shutil.copy(f"{RESULTS_DIR}/test_metrics_full.csv", package_dir)
shutil.copy(f"{RESULTS_DIR}/dm_significance_tests.csv", package_dir)
print(f"Manuscript package assembled at: {package_dir}")
print(os.listdir(package_dir))